In [ ]:
# SHAM_SECRETS_FALLBACK_V3 (2026-10-15)
# بعد ساعات من التدريب فشلت خلية الحفظ بـ NameError: '_sf_orig' / '_sf_creds'
# (الغلاف القديم V1 يعتمد على متغيرات عامة اختفت). هذه الخلية مستقلة تماماً:
# لا تعتمد على أي متغير عام، تضبط KAGGLE_USERNAME/KAGGLE_KEY في البيئة و
# ~/.kaggle/kaggle.json فوراً، وتلتقط أي خطأ من الأغلفة السابقة (حتى NameError)
# فترجع القيم من البيئة. آمنة إن أُعيد تشغيلها.
def _sham_secrets_fallback_v3():
    import glob, json, os

    def load_creds(_glob=glob, _json=json, _os=os):
        u, k = _os.environ.get("KAGGLE_USERNAME"), _os.environ.get("KAGGLE_KEY") or _os.environ.get("KAGGLE_API_TOKEN")
        if u and k:
            return {"username": u, "key": k}
        pats = ["/kaggle/input/sham-runtime-creds/**/kaggle.json", "/kaggle/input/**/kaggle.json",
                _os.path.expanduser("~/.kaggle/kaggle.json")]
        for pat in pats:
            for p in sorted(_glob.glob(pat, recursive=True)):
                try:
                    with open(p) as f:
                        d = _json.load(f)
                    if d.get("username") and d.get("key"):
                        return d
                except Exception:
                    pass
        return {}

    c = load_creds()
    if c:
        os.environ.setdefault("KAGGLE_USERNAME", c["username"])
        if not (os.environ.get("KAGGLE_KEY") or os.environ.get("KAGGLE_API_TOKEN")):
            os.environ["KAGGLE_KEY"] = c["key"]
            if str(c["key"]).startswith("KGAT_"):
                os.environ["KAGGLE_API_TOKEN"] = c["key"]
        try:
            kp = os.path.expanduser("~/.kaggle/kaggle.json")
            if not os.path.exists(kp):
                os.makedirs(os.path.dirname(kp), exist_ok=True)
                with open(kp, "w") as f:
                    json.dump({"username": c["username"], "key": c["key"]}, f)
                os.chmod(kp, 0o600)
        except Exception:
            pass
    print("SHAM_SECRETS_FALLBACK_V3: بيانات كاكل", "جاهزة في البيئة" if c else "غير موجودة")
    try:
        from kaggle_secrets import UserSecretsClient
    except Exception:
        return
    if getattr(UserSecretsClient.get_secret, "_sham_safe3", False):
        return

    def get_secret(self, label, *a, _prev=UserSecretsClient.get_secret, _load=load_creds, **k):
        try:
            return _prev(self, label, *a, **k)
        except BaseException as e:
            if isinstance(e, KeyboardInterrupt):
                raise
            if label == "GITHUB_TOKEN":
                return ""
            if label in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
                d = _load()
                if d:
                    return d["username"] if label == "KAGGLE_USERNAME" else d["key"]
            raise

    get_secret._sham_safe3 = True
    get_secret._sham_safe2 = True  # يمنع V2 من إعادة التغليف لو أعيد تشغيلها
    get_secret._sham_safe = True   # ويمنع V1 كذلك
    UserSecretsClient.get_secret = get_secret

try:
    _sham_secrets_fallback_v3()
except Exception as _sf3_exc:
    print("SHAM_SECRETS_FALLBACK_V3 غير مفعّل:", type(_sf3_exc).__name__)


In [ ]:
# SHAM_QUIET_START_V4 (2026-10-15)
# المالك لاحظ أن اللون الأصفر/البرتقالي للدفتر يظهر حين يكتب الدفتر أخطاء/تحذيرات
# (stderr) ثم يفشل لاحقاً. هذه الخلية تمنع الأخطاء المتوقعة من مصدرها، ولا تغيّر
# أي إعداد للنموذج: (1) تتذكّر السر الذي رفضه كاكل (HTTP 400) فلا تعيد طلبه وتطبع
# الخطأ كل مرة، وتعطي KAGGLE_* من البيئة مباشرة دون طلب الخادم، (2) تضبط
# __main__.__spec__ = None حتى يعمل multiprocessing(spawn) بلا AttributeError،
# (3) تخفّض تحذيرات HuggingFace المتوقعة (طلبات بلا توكن، أشرطة التقدم).
def _sham_quiet_start_v4():
    import os, sys, logging, warnings
    m = sys.modules.get("__main__")
    if m is not None and getattr(m, "__spec__", "missing") == "missing":
        try:
            m.__spec__ = None
        except Exception:
            pass
    os.environ.setdefault("HF_HUB_VERBOSITY", "error")
    os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
    os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    warnings.filterwarnings("ignore", message=".*unauthenticated requests to the HF Hub.*")
    for n in ("huggingface_hub", "datasets", "fsspec", "urllib3"):
        logging.getLogger(n).setLevel(logging.ERROR)
    try:
        from kaggle_secrets import UserSecretsClient
    except Exception:
        return
    if getattr(UserSecretsClient.get_secret, "_sham_safe4", False):
        return
    failed = set()

    def get_secret(self, label, *a, _prev=UserSecretsClient.get_secret, _failed=failed, **k):
        env = {"KAGGLE_USERNAME": os.environ.get("KAGGLE_USERNAME"),
               "KAGGLE_KEY": os.environ.get("KAGGLE_KEY") or os.environ.get("KAGGLE_API_TOKEN")}
        if label in env and env[label]:
            return env[label]            # جاهزة من V3 / sham-runtime-creds: لا طلب ولا خطأ 400
        if label in _failed and label == "GITHUB_TOKEN":
            return ""
        try:
            return _prev(self, label, *a, **k)
        except KeyboardInterrupt:
            raise
        except BaseException:
            _failed.add(label)
            if label == "GITHUB_TOKEN":
                return ""
            raise

    for attr in ("_sham_safe4", "_sham_safe3", "_sham_safe2", "_sham_safe"):
        setattr(get_secret, attr, True)
    UserSecretsClient.get_secret = get_secret

try:
    _sham_quiet_start_v4()
    print("SHAM_QUIET_START_V4: مفعّل")
except Exception as _q4_exc:
    print("SHAM_QUIET_START_V4 غير مفعّل:", type(_q4_exc).__name__)


# شام v2 — تدريب السلالة الجديدة (جسم المانح المنقول + وسائط شام)
دفتر **جديد ومنفصل** (2026-10-26): لا يلمس دفاتر السلالة الحالية ولا مجموعات نقاطها. ينشر إلى مجموعة **جديدة** `sham-v2-checkpoint`.

- أوزان جسم المانح تبقى مجمّدة. **المتدرّب:** محوّلات تكيف منخفضة الرتبة على إسقاطات الانتباه وMLP لتعلّم العربية والأدوات، إضافة إلى صفوف رموز الصورة (8192) والصوت (2048) والرموز الخاصة المرتبطة برأس الإخراج.
- الدفعات: أزواج (صورة/صوت ↔ نص) من خلايا المرحلة الثانية نفسها، + أمثلة أدوات وكود وسؤال-عند-النقص بالصيغة التي يرسلها المحرك، + عربي للمحافظة على اللغة.
- القياس قبل وبعد: أدوات (اسم صحيح + معاملات) عبر التوليد المقيَّد نفسه الذي يعمل في الاستوديو، وربط النص بالوسائط (`measure_link`).
- GPU إن سمح الرصيد، وإلا CPU تلقائياً (المراقب يدفعه GPU أولاً).
- فحص قبل/بعد: قياس مُختبَر خارج عينات التدريب للعربية والكود، وقياس استدعاء الأدوات. النشر الحي مرفوض ما لم تكن الخطوة أكبر من صفر، ويجتاز checkpoint اختبار مُحمّل الإنتاج، ولا تتراجع نتائج العربية أو الأدوات.


### 1) سحب الكود

In [ ]:
import os
import sys
import subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
    print("تم سحب الكود الحقيقي من مستودع GitHub بنجاح.")
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)
    print("الكود موجود بالفعل في هذه الجلسة — تم سحب أي تحديثات جديدة عليه.")

# أمان: git يحفظ رابط الاستنساخ (بما فيه GITHUB_TOKEN) حرفياً داخل
# .git/config -- وهذا المجلد يبقى ضمن نتاج (Output) هذه الجلسة، الذي قد
# يُستخدَم لاحقاً كمُدخَل (Notebook Output) لجلسة أخرى، أو يُشارَك بأي شكل.
# نزع التوكن من الرابط المحفوظ فور نجاح الاستنساخ يمنع تسربه عبر هذا
# المسار تماماً (ثغرة حقيقية اكتشفتها المالكة، 2026-09-21).
subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", "https://github.com/jonsnow-org/Ttbik.git"], check=True)

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
assert os.path.exists(os.path.join(CODE_DIR, "model.py")), f"لم يتم العثور على model.py داخل {CODE_DIR}"
sys.path.insert(0, CODE_DIR)
print("كود ShamSmall الحقيقي جاهز في:", CODE_DIR)


### 2) المكتبات

In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "tokenizers", "soundfile", "safetensors", "transformers", "datasets", "accelerate", "huggingface_hub", "pillow"], check=True)
print("المكتبات جاهزة.")

### 3) الإعداد: مفردات المانح قبل استيراد أي شيء من شام

In [ ]:
import time
SESSION_START = time.time()
import os, json, shutil, random
import sham_v2_train as V
V.setup_env()                                   # SHAM_TEXT_VOCAB: يجب أن يسبق استيراد model.py
import torch
from pathlib import Path
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"الجهاز: {device}")
DONOR = os.environ.get("SHAM_V2_DONOR", V.DEFAULT_DONOR)
from huggingface_hub import hf_hub_download
from text_tokenizer import ShamTextTokenizer
tokenizer = ShamTextTokenizer.load(hf_hub_download(DONOR, "tokenizer.json"))
tokenizer.save("/kaggle/working/sham_small_tokenizer.json")
print(f"مُرمِّز المانح: {tokenizer.vocab_size}")

### 4) النموذج: استئناف من نقطة هذه السلالة، أو التطعيم من المانح في الذاكرة

In [ ]:
from sham_inputs import fetch_dataset, publish_dataset
V_DS = "sham-v2-checkpoint"
OWN = fetch_dataset(V_DS)
_own = sorted(OWN.rglob("final_merged.pt")) if OWN else []
model, start_step = V.prepare_model(DONOR, _own[0] if _own else None, device)
STAGE2_OWN = None
print(f"صفوف الوسائط والرموز الخاصة القابلة للتدريب: {V.freeze_for_media(model):,} (الباقي مجمّد)")

### 5) قياس الأدوات قبل الجلسة (التوليد المقيَّد نفسه الذي في الاستوديو)

In [ ]:
N_TOOL_CASES = int(os.environ.get("SHAM_V2_TOOL_CASES", "24" if device == "cuda" else "8"))
tools_before = V.measure_tools(model, tokenizer, n_cases=N_TOOL_CASES)
print(f"أدوات قبل: صحيح {tools_before['right']}/{tools_before['cases']} | استدعاء سليم {tools_before['valid']} | ask_user {tools_before['asked']}")
arabic_before = V.measure_arabic(model, tokenizer)
print(f"العربية/الكود قبل التدريب: {arabic_before['arabic_script']}/{arabic_before['arabic_cases']} عربي | دمشق={arabic_before['capital_syria']} | كود={arabic_before['python_code']}")


### 6) أزواج الوسائط (جمع وبناء الدفعات: خلايا المرحلة الثانية نفسها)

In [ ]:
from sham_data_sources import Ledger, collect

MAX_IMAGES = 4_000
MAX_AUDIO_SAMPLES = 3_000
# الصور تُجمع بدقة أداة ترميز الصورة نفسها (256) — كانت 64 ثم تُكبَّر فيتعلّم النموذج صوراً ضبابية.
from image_tokenizer import ImageTokenizerConfig as _ITC
COLLECT_IMAGE_SIZE = _ITC().image_size

# كل تشغيل يأخذ بيانات جديدة لم يرها النموذج من قبل: موضع استئناف لكل مصدر + بصمة لكل
# عيّنة، في سجلّين خاصين بالمرحلة الثانية (stage2_image / stage2_audio) يُحفظان في
# /kaggle/working/checkpoints — أضف نتاج هذا الدفتر كمدخل للتشغيل التالي ليكمل بعده.
# (سابقاً: نفس أول 3000 صورة في كل تشغيل، ومصدر Common Voice صار فارغاً على Hugging Face.)
image_ledger = Ledger.load("image", name="stage2_image")
audio_ledger = Ledger.load("audio", name="stage2_audio")
# أزواج ثابتة لقياس ربط النص بالصورة/الصوت قبل الجلسة وبعدها — تُستبعد من التدريب.
from sham_media_link import collect_fixed_pairs
FIXED_PAIRS = collect_fixed_pairs(("image", "audio"), {"image": image_ledger, "audio": audio_ledger}, image_size=COLLECT_IMAGE_SIZE)
image_manifest, image_collect_stats = collect("image", "/kaggle/working/corpus/images", MAX_IMAGES, image_ledger, image_size=COLLECT_IMAGE_SIZE)
audio_manifest, audio_collect_stats = collect("audio", "/kaggle/working/corpus/audio", MAX_AUDIO_SAMPLES, audio_ledger)
# كل المصادر معاً (قرار 2026-10-11): ما جمعته هذه المرحلة مباشرة + عيّنة من دفعات «مصنع الوسائط» (استوديو Lightning)
import sham_media_shards
image_manifest = sham_media_shards.with_factory("image", image_manifest, "/kaggle/working/corpus/image_factory", k=1, limit=4_000) or image_manifest
audio_manifest = sham_media_shards.with_factory("audio", audio_manifest, "/kaggle/working/corpus/audio_factory", k=1, limit=3_000) or audio_manifest
Path("/kaggle/working/checkpoints").mkdir(parents=True, exist_ok=True)
image_ledger.save("/kaggle/working/checkpoints")
audio_ledger.save("/kaggle/working/checkpoints")
assert image_collect_stats.written + audio_collect_stats.written > 0, (
    "لا توجد بيانات جديدة غير مكررة في أي مصدر — المرحلة الثانية رأت كل البيانات المتاحة؛ أضف مصادر جديدة إلى SOURCES."
)
print(f"جاهز: {image_manifest}")
print(f"جاهز: {audio_manifest}")


In [ ]:
import torch
import torch
from PIL import Image
import json as _json
from image_tokenizer import ImageTokenizer, ImageTokenizerConfig
from train_image_tokenizer import train_vqvae as train_image_vqvae, save_tokenizer_checkpoint as save_image_tokenizer
from train_image_tokenizer import load_tokenizer_checkpoint as load_image_tokenizer

# ربط تلقائي بشام: إن كانت أداة ترميز الصورة مدرّبة مسبقاً في مسارها المستقل (CPU)
# ومُضافة كمُدخل (Add Input) لهذا الدفتر، نستخدمها كما هي بدل تدريب أداة
# جديدة صغيرة من الصفر — لا حاجة لمعرفة اسم المجموعة أو مسارها، البحث تلقائي.
# يقبل المسارين تلقائياً بلا تعارض: أي مجموعة مُضافة تحوي image_tokenizer.pt
# (مسار الترميز المستقل، أو مجموعة مدمجة) — يُختار الأكثر تدريباً والمتوافق
# مع النموذج (tokenizer_select.py)، وإن لم يوجد شيء صالح يُدرَّب هنا من جديد.
from tokenizer_select import select_pretrained_tokenizer
print("البحث عن أداة ترميز جاهزة (image):")
# عند استئناف ناتج هذه المرحلة: نفس أداة الترميز التي تدرّب معها النموذج أولاً (أداة أحدث
# تعطي رموزاً مختلفة لنفس الصورة/الصوت)، وإلا الأكثر تدريباً بين المدخلات.
_picked_image = (select_pretrained_tokenizer("image", STAGE2_OWN) if STAGE2_OWN else None) or select_pretrained_tokenizer("image")
_pretrained_image = [_picked_image[2]] if _picked_image else []
if _picked_image:
    image_tokenizer, _image_step = _picked_image[0], _picked_image[1]
    image_tokenizer_cfg = image_tokenizer.cfg
    print(f"✅ استُخدمت أداة ترميز الصورة المدرّبة مسبقاً: {_pretrained_image[-1]} (خطوة {_image_step})")
    save_image_tokenizer("/kaggle/working/image_tokenizer.pt", image_tokenizer, step=_image_step)
else:
    print("لم يُعثر على أداة ترميز الصورة مدرّبة ضمن المدخلات — تدريب أداة جديدة هنا.")


    image_tokenizer_cfg = ImageTokenizerConfig(
        image_size=64, base_channels=32, channel_multipliers=(1, 2, 4), code_dim=64,  # حجم بداية آمن
        # num_codes يبقى ثابتاً على القيمة المعمارية الحقيقية (8192) — لا يُغيَّر أبداً، فالنموذج الرئيسي يحجز لها هذا المدى بالضبط.
    )
    image_tokenizer = ImageTokenizer(image_tokenizer_cfg)
    print(f"عدد رموز كل صورة: {image_tokenizer_cfg.tokens_per_image} (شبكة {image_tokenizer_cfg.latent_grid_size}x{image_tokenizer_cfg.latent_grid_size})")

    # تحميل الصور الحقيقية إلى مصفوفة واحدة في الذاكرة (مناسب لبضعة آلاف صورة بحجم 64x64)
    image_root = Path(image_manifest).parent
    images_list = []
    with open(image_manifest, encoding="utf-8") as f:
        for line in f:
            record = _json.loads(line)
            img = Image.open(image_root / record["image"]).convert("RGB").resize((image_tokenizer_cfg.image_size, image_tokenizer_cfg.image_size))
            tensor = torch.tensor(list(img.getdata()), dtype=torch.float32).view(image_tokenizer_cfg.image_size, image_tokenizer_cfg.image_size, 3)
            images_list.append(tensor.permute(2, 0, 1) / 127.5 - 1.0)
    real_images = torch.stack(images_list, dim=0)
    print(f"عدد الصور الحقيقية المحمّلة للتدريب: {real_images.shape[0]:,}")

    image_vqvae_stats = train_image_vqvae(image_tokenizer, real_images, num_epochs=10, batch_size=32, device=device, log_every=2)
    print(f"آخر خسارة إعادة بناء+VQ حقيقية: {image_vqvae_stats.epoch_losses[-1]:.4f} | "
          f"استخدام القاموس (codebook): {image_vqvae_stats.final_codebook_usage}/{image_vqvae_stats.codebook_size}")

    save_image_tokenizer("/kaggle/working/image_tokenizer.pt", image_tokenizer, step=len(image_vqvae_stats.epoch_losses))
    print("تم حفظ أداة ترميز الصورة الحقيقية المدرَّبة.")

# الترميز داخل MultimodalCollator يجري على CPU (الصور/المقاطع تُحمَّل على CPU)،
# ثم تُنقل الدفعات الجاهزة إلى GPU للنموذج الرئيسي — لذا تبقى أداة الترميز على
# CPU وفي وضع التقييم (مجمّدة) في الحالتين: المدرّبة مسبقاً والمدرّبة هنا.
image_tokenizer = image_tokenizer.to("cpu").train(False)

In [ ]:
import soundfile as sf
from mel_spectrogram import waveform_to_mel_segments, waveform_to_mel_spectrogram
from audio_tokenizer import AudioTokenizer, AudioTokenizerConfig
from train_audio_tokenizer import train_vqvae as train_audio_vqvae, save_tokenizer_checkpoint as save_audio_tokenizer
from train_audio_tokenizer import load_tokenizer_checkpoint as load_audio_tokenizer

# ربط تلقائي بشام: إن كانت أداة ترميز الصوت مدرّبة مسبقاً في مسارها المستقل (CPU)
# ومُضافة كمُدخل (Add Input) لهذا الدفتر، نستخدمها كما هي بدل تدريب أداة
# جديدة صغيرة من الصفر — لا حاجة لمعرفة اسم المجموعة أو مسارها، البحث تلقائي.
# يقبل المسارين تلقائياً بلا تعارض: أي مجموعة مُضافة تحوي audio_tokenizer.pt
# (مسار الترميز المستقل، أو مجموعة مدمجة) — يُختار الأكثر تدريباً والمتوافق
# مع النموذج (tokenizer_select.py)، وإن لم يوجد شيء صالح يُدرَّب هنا من جديد.
from tokenizer_select import select_pretrained_tokenizer
print("البحث عن أداة ترميز جاهزة (audio):")
# عند استئناف ناتج هذه المرحلة: نفس أداة الترميز التي تدرّب معها النموذج أولاً (أداة أحدث
# تعطي رموزاً مختلفة لنفس الصورة/الصوت)، وإلا الأكثر تدريباً بين المدخلات.
_picked_audio = (select_pretrained_tokenizer("audio", STAGE2_OWN) if STAGE2_OWN else None) or select_pretrained_tokenizer("audio")
_pretrained_audio = [_picked_audio[2]] if _picked_audio else []
if _picked_audio:
    audio_tokenizer, _audio_step = _picked_audio[0], _picked_audio[1]
    audio_tokenizer_cfg = audio_tokenizer.cfg
    print(f"✅ استُخدمت أداة ترميز الصوت المدرّبة مسبقاً: {_pretrained_audio[-1]} (خطوة {_audio_step})")
    save_audio_tokenizer("/kaggle/working/audio_tokenizer.pt", audio_tokenizer, step=_audio_step)
else:
    print("لم يُعثر على أداة ترميز الصوت مدرّبة ضمن المدخلات — تدريب أداة جديدة هنا.")

    audio_tokenizer_cfg = AudioTokenizerConfig()  # الحجم الافتراضي مناسب هنا (num_codes يطابق AUDIO_VOCAB_SIZE أصلاً)
    audio_tokenizer = AudioTokenizer(audio_tokenizer_cfg)
    print(f"عدد رموز كل مقطع صوتي: {audio_tokenizer_cfg.tokens_per_segment}")

    audio_root = Path(audio_manifest).parent
    mels_list = []
    with open(audio_manifest, encoding="utf-8") as f:
        for line in f:
            record = _json.loads(line)
            waveform, sample_rate = sf.read(str(audio_root / record["audio"]), dtype="float32", always_2d=False)
            if waveform.ndim > 1:
                waveform = waveform.mean(axis=1)
            # كل مقاطع المقطع الصوتي (حتى 6)، لا اقتطاع 2.56 ثانية واحدة من منتصفه (إصلاح 2026-10-11)
            mels_list.extend(waveform_to_mel_segments(
                torch.from_numpy(waveform), sample_rate, audio_tokenizer_cfg.n_mels, audio_tokenizer_cfg.segment_frames, 6
            ))
    real_mels = torch.stack(mels_list, dim=0)
    print(f"عدد المقاطع الصوتية الحقيقية المحمّلة للتدريب: {real_mels.shape[0]:,}")

    audio_vqvae_stats = train_audio_vqvae(audio_tokenizer, real_mels, num_epochs=10, batch_size=32, device=device, log_every=2)
    print(f"آخر خسارة إعادة بناء+VQ حقيقية: {audio_vqvae_stats.epoch_losses[-1]:.4f} | "
          f"استخدام القاموس (codebook): {audio_vqvae_stats.final_codebook_usage}/{audio_vqvae_stats.codebook_size}")

    save_audio_tokenizer("/kaggle/working/audio_tokenizer.pt", audio_tokenizer, step=len(audio_vqvae_stats.epoch_losses))
    print("تم حفظ أداة ترميز الصوت الحقيقية المدرَّبة.")

# الترميز داخل MultimodalCollator يجري على CPU (الصور/المقاطع تُحمَّل على CPU)،
# ثم تُنقل الدفعات الجاهزة إلى GPU للنموذج الرئيسي — لذا تبقى أداة الترميز على
# CPU وفي وضع التقييم (مجمّدة) في الحالتين: المدرّبة مسبقاً والمدرّبة هنا.
audio_tokenizer = audio_tokenizer.to("cpu").train(False)

# تأكيد صريح: نفس أداتي الترميز المحفوظتين مع نقطة هذه المرحلة (لا إعادة تهيئة).
from sham_media_link import confirm_saved_tokenizers
confirm_saved_tokenizers(image_tokenizer, audio_tokenizer, STAGE2_OWN)

In [ ]:
from dataset import ImageCaptionDataset, AudioTranscriptDataset, MultimodalCollator, AudioMultimodalCollator

image_dataset = ImageCaptionDataset(image_manifest, image_size=image_tokenizer_cfg.image_size)
audio_dataset = AudioTranscriptDataset(audio_manifest, n_mels=audio_tokenizer_cfg.n_mels, segment_frames=audio_tokenizer_cfg.segment_frames, max_segments=4)
print(f"عدد أزواج (صورة، تعليق): {len(image_dataset)}")
print(f"عدد أزواج (صوت، نص): {len(audio_dataset)}")
from sham_media_link import data_report
data_report("الصورة", image_collect_stats, image_dataset)
data_report("الصوت", audio_collect_stats, audio_dataset)

image_collator = MultimodalCollator(tokenizer, image_tokenizer, both_directions=True)
audio_collator = AudioMultimodalCollator(tokenizer, audio_tokenizer, both_directions=True)

IMAGE_BATCH_SIZE = 4
AUDIO_BATCH_SIZE = 4

multimodal_batches = []
for start in range(0, len(image_dataset) - (len(image_dataset) % IMAGE_BATCH_SIZE), IMAGE_BATCH_SIZE):
    batch = [image_dataset[i] for i in range(start, start + IMAGE_BATCH_SIZE)]
    multimodal_batches.append(image_collator(batch))
for start in range(0, len(audio_dataset) - (len(audio_dataset) % AUDIO_BATCH_SIZE), AUDIO_BATCH_SIZE):
    batch = [audio_dataset[i] for i in range(start, start + AUDIO_BATCH_SIZE)]
    multimodal_batches.append(audio_collator(batch))

import random
random.Random(0).shuffle(multimodal_batches)  # لا نريد كل الصور أولاً ثم كل الصوت — نخلطهما فعلياً
print(f"عدد دفعات (نص+صورة) و(نص+صوت) الحقيقية الجاهزة: {len(multimodal_batches):,} "
      f"(كل صورة/مقطع يُدرَّب بالاتجاهين معاً: توليد + فهم).")

assert multimodal_batches, "لا توجد دفعات كافية — كبّر MAX_IMAGES/MAX_AUDIO_SAMPLES في خلية جمع البيانات."


### 7) أمثلة الأدوات والكود والعربي

In [ ]:
SEED = int(time.time()) // 86400                 # بذرة اليوم: أمثلة جديدة كل جلسة، وغير بذرة القياس الثابتة (4242)
tool_b = V.tool_batches(tokenizer, 1600, 2, seed=SEED)
ar_pairs = []
try:
    from datasets import load_dataset
    for name, split, qf, af, flt in (("arbml/CIDAR", "train", "instruction", "output", None), ("CohereLabs/aya_dataset", "train", "inputs", "targets", "Arabic")):
        ds = load_dataset(name, split=split, streaming=True)
        if flt:
            ds = ds.filter(lambda r: flt in (r.get("language") or ""))
        for i, r in enumerate(ds.skip(SEED % 5000)):
            q, a = (r.get(qf) or "").strip(), (r.get(af) or "").strip()
            if q and a:
                ar_pairs.append((q[:600], a[:900]))
            if i >= 1500:
                break
except Exception as _exc:
    print(f"العربي: تخطّي ({type(_exc).__name__}: {str(_exc)[:100]})")
# حقائق تدريبية فقط: يتعلم شام هويته من البيانات، وليس من رد ثابت وقت التشغيل.
ar_pairs.extend(V.identity_grounding_pairs())
ar_b = V.arabic_pairs_to_batches(tokenizer, ar_pairs, 2) if ar_pairs else []
print(f"دفعات: وسائط {len(multimodal_batches):,} | أدوات {len(tool_b):,} | عربي {len(ar_b):,}")
mix = list(multimodal_batches) + tool_b + ar_b
random.Random(SEED).shuffle(mix)

### 8) قياس الربط قبل الجلسة ثم التدريب (جلسة GPU أقصر حفاظاً على الرصيد)

In [ ]:
from sham_media_link import measure_link, link_report
model.to(device)
_media_toks = {"image": image_tokenizer, "audio": audio_tokenizer}
link_before = measure_link(model, tokenizer, _media_toks, FIXED_PAIRS, device)
link_report(link_before)
HOURS = float(os.environ.get("SHAM_V2_HOURS", "3.5" if device == "cuda" else "8.0"))
HOURS = min(HOURS, max(0.5, (11.0 * 3600 - (time.time() - SESSION_START) - 40 * 60) / 3600))
losses, final_step = V.run_session(model, start_step, mix, device, HOURS)
print(f"انتهى: {len(losses):,} خطوة — الخطوة النهائية {final_step:,}")
if losses:
    print(f"الخسارة: أول 10 = {sum(losses[:10]) / min(10, len(losses)):.4f} | آخر 10 = {sum(losses[-10:]) / min(10, len(losses)):.4f}")
link_after = measure_link(model, tokenizer, _media_toks, FIXED_PAIRS, device)
link_report(link_before, link_after)

### 9) قياس الأدوات بعد الجلسة

In [ ]:
tools_after = V.measure_tools(model, tokenizer, n_cases=N_TOOL_CASES)
print(f"أدوات بعد: صحيح {tools_after['right']}/{tools_after['cases']} | استدعاء سليم {tools_after['valid']} | ask_user {tools_after['asked']}")
for r in tools_after["rows"][:6]:
    print("  ", r)
arabic_after = V.measure_arabic(model, tokenizer)
print(f"العربية/الكود بعد التدريب: {arabic_after['arabic_script']}/{arabic_after['arabic_cases']} عربي | دمشق={arabic_after['capital_syria']} | كود={arabic_after['python_code']}")
for r in arabic_after['rows']:
    print('  ', r)


### 10) الحفظ مع بصمة المُرمِّز، والتحقق من أنها تُقبل، ثم النشر إلى `sham-v2-checkpoint`

In [ ]:
from checkpoint import save_checkpoint
from sham_tok_id import stamp, resolve
import json as _json
upload_dir = Path("/kaggle/working/for_dataset_upload")
if upload_dir.exists():
    shutil.rmtree(upload_dir)
upload_dir.mkdir(parents=True)
extra = {"native_tokenizer": True, **stamp(tokenizer)}
save_checkpoint(upload_dir / "final_merged.pt", model, final_step, extra=extra)
shutil.copy2("/kaggle/working/sham_small_tokenizer.json", upload_dir / "sham_small_tokenizer.json")
for _f in ("image_tokenizer.pt", "audio_tokenizer.pt"):
    shutil.copy2(Path("/kaggle/working") / _f, upload_dir / _f)
for _ledger in Path("/kaggle/working/checkpoints").glob("*_ledger.json"):
    shutil.copy2(_ledger, upload_dir / _ledger.name)
_saved = torch.load(upload_dir / "final_merged.pt", map_location="cpu", weights_only=False)
_path, _note = resolve(_saved.get("extra"), upload_dir / "sham_small_tokenizer.json")
assert _path == str(upload_dir / "sham_small_tokenizer.json") and _note.startswith("ok"), _note        # serve.py would accept it
del _saved
_hist = []
if OWN:
    _hp = sorted(OWN.rglob("v2_history.json"))
    if _hp:
        _hist = _json.loads(_hp[0].read_text(encoding="utf-8"))
_hist.append({"step": final_step, "device": device, "donor": DONOR, "tools_before": {k: v for k, v in tools_before.items() if k != "rows"},
              "tools_after": {k: v for k, v in tools_after.items() if k != "rows"}, "arabic_before": {k: v for k, v in arabic_before.items() if k != "rows"}, "arabic_after": {k: v for k, v in arabic_after.items() if k != "rows"}, "train_loss_last": losses[-20:] if losses else []})
(upload_dir / "v2_history.json").write_text(_json.dumps(_hist, ensure_ascii=False, default=float), encoding="utf-8")
print("بصمة المُرمِّز:", _note)
publish_dataset(upload_dir, V_DS, f"sham v2 at step {final_step:,}")

In [ ]:
# Ask the restarter to run right after this session ends (added 2026-10-16; deployment only, no training change).
try:
    import sham_trigger_restarter
    sham_trigger_restarter.trigger()
except Exception as _exc:
    print(f"sham_trigger_restarter skipped: {_exc}")
